## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_curve, classification_report

# Plot styling
DARK  = '#1a1a2e'
POS   = '#e05c5c'   # disease present
NEG   = '#4a90d9'   # no disease
BG    = '#f9f9f9'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.facecolor': BG,
    'figure.facecolor': 'white',
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
})

print('Libraries loaded successfully.')

## 2. Data Loading and Preparation

In [ ]:
# Load dataset
df = pd.read_csv('https://raw.githubusercontent.com/Bizarroxela/Heart_Disease_Analysis/refs/heads/main/data/heart_disease_targets.csv')

print('Raw shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Check for '?' missing value encoding
for col in df.columns:
    mask = df[col].astype(str).str.strip() == '?'
    if mask.sum() > 0:
        print(f"'{col}' contains {mask.sum()} '?' values")

# Replace '?' with NaN and convert to numeric
df = df.replace('?', np.nan)
df['ca']   = pd.to_numeric(df['ca'])
df['thal'] = pd.to_numeric(df['thal'])

# Impute missing values with median
df['ca']   = df['ca'].fillna(df['ca'].median())
df['thal'] = df['thal'].fillna(df['thal'].median())

print('Missing values after imputation:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())

In [ ]:
# Binarize target: 0 = no disease, 1 = disease present (any severity)
df['target'] = (df['target'] > 0).astype(int)

print('Target distribution after binarization:')
print(df['target'].value_counts())
print(f"\nClass balance: {df['target'].mean():.1%} positive")

# Feature matrix and target vector
X = df.drop('target', axis=1)
y = df['target']

print(f'\nFeatures: {X.shape[1]}, Samples: {X.shape[0]}')

In [ ]:
# Summary statistics
df.describe().round(2)

## 3. Exploratory Data Analysis

In [ ]:
# Figure 1 — Continuous feature distributions by outcome
cont_cols   = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
cont_labels = ['Age (years)', 'Resting BP (mm Hg)', 'Cholesterol (mg/dl)', 'Max Heart Rate', 'ST Depression']

fig, axes = plt.subplots(1, 5, figsize=(18, 4.5))
fig.suptitle('Figure 1 — Continuous Feature Distributions by Outcome (n=303)',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)

for ax, col, label in zip(axes, cont_cols, cont_labels):
    for val, color, lbl in [(0, NEG, 'No Disease'), (1, POS, 'Disease Present')]:
        ax.hist(df[df['target'] == val][col], bins=20, alpha=0.65,
                color=color, label=lbl, edgecolor='white', linewidth=0.4)
    ax.set_title(label)
    ax.set_ylabel('Count' if ax is axes[0] else '')

axes[-1].legend(framealpha=0.8, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2 — Categorical feature distributions
cat_info = [
    ('cp',      'Chest Pain Type',  {1:'Typical\nAngina', 2:'Atypical\nAngina', 3:'Non-Anginal', 4:'Asymptomatic'}),
    ('sex',     'Sex',              {0:'Female', 1:'Male'}),
    ('exang',   'Exercise Angina',  {0:'No', 1:'Yes'}),
    ('thal',    'Thal',             {3:'Normal', 6:'Fixed\nDefect', 7:'Reversible\nDefect'}),
    ('slope',   'ST Slope',         {1:'Upsloping', 2:'Flat', 3:'Downsloping'}),
    ('restecg', 'Resting ECG',      {0:'Normal', 1:'ST-T\nAbnorm.', 2:'LV\nHypertrophy'}),
    ('fbs',     'Fasting BS>120',   {0:'No', 1:'Yes'}),
    ('ca',      'Major Vessels',    {0:'0', 1:'1', 2:'2', 3:'3'}),
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Figure 2 — Categorical Feature % with Disease Present',
             fontsize=13, fontweight='bold', color=DARK, y=1.01)

for ax, (col, title, label_map) in zip(axes.flat, cat_info):
    ct = pd.crosstab(df[col], df['target'], normalize='index') * 100
    for c in [0, 1]:
        if c not in ct.columns:
            ct[c] = 0
    ct = ct[[0, 1]]
    x = np.arange(len(ct))
    ax.bar(x, ct[0].values, 0.55, color=NEG, alpha=0.8, label='No Disease')
    ax.bar(x, ct[1].values, 0.55, bottom=ct[0].values, color=POS, alpha=0.8, label='Disease')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([label_map.get(v, str(v)) for v in ct.index], fontsize=8)
    ax.set_ylim(0, 115)
    ax.set_ylabel('% of Category')
    ax.axhline(50, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
    for i, (v0, v1) in enumerate(zip(ct[0].values, ct[1].values)):
        ax.text(i, v0 + v1/2, f'{v1:.0f}%', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')

axes[0, 0].legend(fontsize=9, framealpha=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 3 — Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle('Figure 3 — Feature Correlation Matrix', fontsize=13, fontweight='bold', color=DARK)

corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(220, 10, as_cmap=True),
            center=0, vmin=-1, vmax=1, annot=True, fmt='.2f',
            annot_kws={'size': 9}, linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

print('\nCorrelations with target (sorted):')
print(df.corr()['target'].sort_values(ascending=False).round(3))

## 4. Model Training and Cross-Validated Comparison

In [ ]:
# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define models (scale-sensitive models wrapped in Pipeline)
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest':     RandomForestClassifier(n_estimators=300, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'SVM':               Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=True, random_state=42))
    ]),
    'KNN':               Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier())
    ]),
}

# Run cross-validated scoring
metrics = ['accuracy', 'recall', 'precision', 'f1', 'roc_auc']
results = {}

for name, model in models.items():
    scores = {m: cross_val_score(model, X, y, cv=cv, scoring=m) for m in metrics}
    results[name] = scores
    print(f"{name:25s}  Acc={scores['accuracy'].mean():.3f}  "
          f"Recall={scores['recall'].mean():.3f}  "
          f"F1={scores['f1'].mean():.3f}  "
          f"AUC={scores['roc_auc'].mean():.3f}")

In [ ]:
# Figure 4 — Model comparison bar chart
model_names  = list(results.keys())
acc_means    = [results[m]['accuracy'].mean()  for m in model_names]
recall_means = [results[m]['recall'].mean()    for m in model_names]
f1_means     = [results[m]['f1'].mean()        for m in model_names]
auc_means    = [results[m]['roc_auc'].mean()   for m in model_names]

x = np.arange(len(model_names))
width = 0.2
COLORS = ['#4a90d9', '#e05c5c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle('Figure 4 — Model Comparison: 5-Fold Cross-Validated Metrics',
             fontsize=13, fontweight='bold', color=DARK)

for i, (metric, vals) in enumerate(zip(
        ['Accuracy', 'Recall', 'F1 Score', 'AUC-ROC'],
        [acc_means, recall_means, f1_means, auc_means])):
    ax.bar(x + i * width, vals, width, label=metric, color=COLORS[i], alpha=0.85, edgecolor='white')

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylim(0.65, 0.98)
ax.set_ylabel('Score')
ax.legend(loc='lower right', fontsize=9)
ax.axhline(0.90, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning

In [ ]:
# Random Forest tuning
rf_params = {
    'n_estimators': [200, 300, 400],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params,
                       cv=cv, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X, y)
print(f'RF   best params: {rf_grid.best_params_}  |  AUC: {rf_grid.best_score_:.3f}')

# Gradient Boosting tuning
gb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4]
}
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42), gb_params,
                       cv=cv, scoring='roc_auc', n_jobs=-1)
gb_grid.fit(X, y)
print(f'GB   best params: {gb_grid.best_params_}  |  AUC: {gb_grid.best_score_:.3f}')

# Logistic Regression tuning
lr_params = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l1', 'l2'],
    'clf__solver': ['liblinear']
}
lr_pipe = Pipeline([('scaler', StandardScaler()),
                    ('clf', LogisticRegression(max_iter=1000, random_state=42))])
lr_grid = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring='roc_auc', n_jobs=-1)
lr_grid.fit(X, y)
print(f'LR   best params: {lr_grid.best_params_}  |  AUC: {lr_grid.best_score_:.3f}')

In [ ]:
# Soft-voting ensemble from tuned models
ensemble = VotingClassifier(
    estimators=[
        ('rf', rf_grid.best_estimator_),
        ('gb', gb_grid.best_estimator_),
        ('lr', lr_grid.best_estimator_),
    ],
    voting='soft'
)

ens_auc = cross_val_score(ensemble, X, y, cv=cv, scoring='roc_auc').mean()
ens_rec = cross_val_score(ensemble, X, y, cv=cv, scoring='recall').mean()
ens_acc = cross_val_score(ensemble, X, y, cv=cv, scoring='accuracy').mean()
print(f'Ensemble (soft vote):  Acc={ens_acc:.3f}  Recall={ens_rec:.3f}  AUC={ens_auc:.3f}')

## 6. ROC Curves and Confusion Matrices

In [ ]:
# Figure 5 — ROC curves
named_models = [
    ('Logistic Regression', lr_grid.best_estimator_),
    ('Random Forest',       rf_grid.best_estimator_),
    ('Gradient Boosting',   gb_grid.best_estimator_),
    ('SVM',                 Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))])),
    ('KNN',                 Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())])),
    ('Ensemble (Tuned)',    ensemble),
]
roc_colors = ['#4a90d9', '#e05c5c', '#2ecc71', '#f39c12', '#9b59b6', '#1a1a2e']

fig, ax = plt.subplots(figsize=(8, 7))
fig.suptitle('Figure 5 — ROC Curves (5-Fold Cross-Validation)',
             fontsize=13, fontweight='bold', color=DARK)

for (name, model), color in zip(named_models, roc_colors):
    y_prob = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
    fpr, tpr, _ = roc_curve(y, y_prob)
    from sklearn.metrics import auc
    auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc_val:.3f})')

ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, alpha=0.6)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9, loc='lower right', framealpha=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 6 — Confusion matrices for top 3 models
top3 = [('Logistic Regression', lr_grid.best_estimator_),
        ('Random Forest',       rf_grid.best_estimator_),
        ('SVM',                 Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, random_state=42))]))]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.suptitle('Figure 6 — Confusion Matrices: Top 3 Models (CV Predictions)',
             fontsize=13, fontweight='bold', color=DARK)

for ax, (name, model) in zip(axes, top3):
    y_pred = cross_val_predict(model, X, y, cv=cv)
    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'],
                linewidths=0.5, linecolor='white', cbar=False,
                annot_kws={'size': 13})
    tn, fp, fn, tp = cm.ravel()
    recall = tp / (tp + fn)
    spec   = tn / (tn + fp)
    ax.set_title(f'{name}\nRecall={recall:.3f} | Specificity={spec:.3f}', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    print(f'{name}: Recall={recall:.3f}, Specificity={spec:.3f}')

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

In [ ]:
# Figure 7 — RF feature importance + LR coefficients
rf_best = rf_grid.best_estimator_
lr_best = lr_grid.best_estimator_

rf_best.fit(X, y)
lr_best.fit(X, y)

imp   = pd.Series(rf_best.feature_importances_, index=X.columns).sort_values(ascending=True)
coefs = pd.Series(lr_best.named_steps['clf'].coef_[0], index=X.columns).sort_values(ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 7 — Feature Importance: Random Forest & Logistic Regression',
             fontsize=13, fontweight='bold', color=DARK)

colors_rf = [POS if v >= imp.median() else NEG for v in imp.values]
ax1.barh(imp.index, imp.values, color=colors_rf, edgecolor='white')
ax1.set_title('Random Forest Feature Importances')
ax1.set_xlabel('Importance Score')
ax1.axvline(imp.median(), color='gray', linestyle='--', linewidth=0.9, alpha=0.6)

colors_lr = [POS if v > 0 else NEG for v in coefs.values]
ax2.barh(coefs.index, coefs.values, color=colors_lr, edgecolor='white')
ax2.set_title('Logistic Regression Coefficients\n(red = increases risk, blue = decreases risk)')
ax2.set_xlabel('Coefficient Value')
ax2.axvline(0, color='gray', linewidth=1, alpha=0.6)

plt.tight_layout()
plt.show()

print('\nTop 5 RF features:')
print(imp.sort_values(ascending=False).head())
print('\nTop 5 LR features (by abs coefficient):')
print(coefs.abs().sort_values(ascending=False).head())